## 09 - Treino final multiseed e seleção operacional por validação (v3)
Este notebook treina a família já congelada, YOLO11n-OBB, nas seeds
20260823, 20260824 e 20260825. Ele não escolhe novamente a família e não lê
predições do teste sintético nem dados ASTA.
A seed 20260823 pode ser reutilizada do Notebook 08 somente porque arquitetura,
peso inicial, dados, hiperparâmetros e seed são idênticos e todos os hashes são
verificados. As outras duas seeds são treinos independentes e retomáveis.
Ao final, os três best.pt são avaliados somente na validação. Um checkpoint
operacional é congelado para uso externo, mas o teste sintético futuro avaliará os
três checkpoints em um único evento e reportará variabilidade entre seeds. Nenhuma
seed poderá ser escolhida depois de observar o teste.

In [1]:
import json, platform, sys, torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU obrigatória. No Colab, selecione Ambiente de execução > Alterar tipo de '
        'ambiente > GPU e execute novamente a partir desta célula.'
    )
props=torch.cuda.get_device_properties(0)
livre,total=torch.cuda.mem_get_info(0)
relatorio_gpu={
    'status':'GPU_APROVADA','gpu':torch.cuda.get_device_name(0),
    'cuda':torch.version.cuda,'torch':torch.__version__,
    'vram_total_gb':round(total/1024**3,2),'vram_livre_gb':round(livre/1024**3,2),
    'python':sys.version.split()[0],'plataforma':platform.platform()
}
assert total>=8*1024**3,'GPU com menos de 8 GiB não foi aprovada para este protocolo.'
print(json.dumps(relatorio_gpu,indent=2,ensure_ascii=False))

{
  "status": "GPU_APROVADA",
  "gpu": "Tesla T4",
  "cuda": "12.8",
  "torch": "2.11.0+cu128",
  "vram_total_gb": 14.56,
  "vram_livre_gb": 14.46,
  "python": "3.13.15",
  "plataforma": "Linux-6.6.122+-x86_64-with-glibc2.35"
}


## 1. Ambiente fixado e restauração pelo TAR
A restauração é idêntica à aprovada no Notebook 08: um único TAR, verificação de
SHA-256 e extração local. Execute em GPU desde a primeira célula.

In [2]:
%pip install -q ultralytics==8.4.127 opencv-python-headless pandas pyyaml tqdm

from google.colab import drive
from pathlib import Path
import hashlib, json, math, os, shutil, tarfile, tempfile, time
import cv2
import numpy as np
import pandas as pd
import yaml
from tqdm.auto import tqdm

def log_etapa(mensagem):
    print(f"[{time.strftime('%H:%M:%S')}] {mensagem}",flush=True)

def sha256_arquivo(caminho,bloco=8*1024*1024):
    h=hashlib.sha256()
    with open(caminho,'rb') as arq:
        for parte in iter(lambda:arq.read(bloco),b''):h.update(parte)
    return h.hexdigest()

def sha256_com_progresso(caminho,descricao,bloco=8*1024*1024):
    caminho=Path(caminho);total=caminho.stat().st_size;h=hashlib.sha256()
    with caminho.open('rb') as arq,tqdm(total=total,desc=descricao,unit='B',unit_scale=True) as barra:
        for parte in iter(lambda:arq.read(bloco),b''):
            h.update(parte);barra.update(len(parte))
    return h.hexdigest()

def copiar_arquivo_com_progresso(origem,destino,descricao,bloco=8*1024*1024):
    origem,destino=Path(origem),Path(destino);destino.parent.mkdir(parents=True,exist_ok=True)
    total=origem.stat().st_size
    with origem.open('rb') as entrada,destino.open('wb') as saida, \
         tqdm(total=total,desc=descricao,unit='B',unit_scale=True) as barra:
        for parte in iter(lambda:entrada.read(bloco),b''):
            saida.write(parte);barra.update(len(parte))

def copiar_arvore_com_progresso(origem,destino,descricao):
    origem,destino=Path(origem),Path(destino)
    arquivos=sorted(p for p in origem.rglob('*') if p.is_file())
    for arquivo in tqdm(arquivos,desc=descricao,unit='arq'):
        alvo=destino/arquivo.relative_to(origem);alvo.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(arquivo,alvo)

drive.mount('/content/drive',force_remount=True)
BACKUP_DIR=Path('/content/drive/MyDrive/tcc-satellite-streaks')
RUN_ID=(BACKUP_DIR/'data/synthetic_runs/LATEST_RUN_V3.txt').read_text(encoding='utf-8').strip()
PACOTES_DRIVE=BACKUP_DIR/'data/yolo_packages'/RUN_ID
contrato_pacote=json.loads((PACOTES_DRIVE/'contrato_pacote_yolo_v3.json').read_text(encoding='utf-8'))
assert contrato_pacote['run_id']==RUN_ID
pacote_remoto=PACOTES_DRIVE/contrato_pacote['arquivo']
pacote_local=Path('/content')/contrato_pacote['arquivo']

if not pacote_local.exists():
    copiar_arquivo_com_progresso(pacote_remoto,pacote_local,'Restaurando TAR do dataset')
hash_pacote=sha256_com_progresso(pacote_local,'Verificando TAR local')
assert hash_pacote==contrato_pacote['sha256_pacote']

raiz_runs=Path('/content/data/synthetic_runs');raiz_runs.mkdir(parents=True,exist_ok=True)
RUN=raiz_runs/RUN_ID
if RUN.exists():
    assert (RUN/'manifest_geracao_v3.csv').is_file()
    assert sha256_arquivo(RUN/'manifest_geracao_v3.csv')==contrato_pacote['sha256_manifest']
    print('RUN local íntegro reutilizado.')
else:
    temp=Path(tempfile.mkdtemp(prefix='extracao_yolo_',dir='/content'))
    try:
        with tarfile.open(pacote_local,'r') as tar:
            membros=tar.getmembers()
            assert len(membros)==contrato_pacote['membros']
            for membro in membros:
                partes=Path(membro.name).parts
                assert partes and partes[0]==RUN_ID and '..' not in partes
                assert not membro.issym() and not membro.islnk()
            for membro in tqdm(membros,desc='Extraindo dataset local',unit='arq'):
                tar.extract(membro,path=temp,filter='data')
        shutil.move(str(temp/RUN_ID),str(RUN))
    finally:
        if temp.exists():shutil.rmtree(temp)

df=pd.read_csv(RUN/'manifest_geracao_v3.csv')
HASH_MANIFEST=sha256_arquivo(RUN/'manifest_geracao_v3.csv')
validacao=json.loads((RUN/'resumo_validacao_v3.json').read_text(encoding='utf-8'))
assert len(df)==6000 and HASH_MANIFEST==contrato_pacote['sha256_manifest']
assert validacao.get('aprovado_para_treino') is True
for split in ['train','val','test']:
    esperado=set(df.loc[df.split==split,'id'])
    assert {p.stem for p in (RUN/'images'/split).glob('*.png')}==esperado
    assert {p.stem for p in (RUN/'labels'/split).glob('*.txt')}==esperado

dataset_yaml=RUN/'dataset_yolo_selection_v3.yaml'
dataset_yaml.write_text(yaml.safe_dump({
    'path':str(RUN.resolve()),'train':'images/train','val':'images/val','test':'images/test',
    'names':{0:'satellite_streak'}
},sort_keys=False),encoding='utf-8')
print(json.dumps({
    'status':'DATASET_GPU_LOCAL_APROVADO','run_id':RUN_ID,
    'sha256_manifest':HASH_MANIFEST,'amostras':len(df),
    'pacote_sha256':hash_pacote
},indent=2,ensure_ascii=False))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 6.2 MB/s eta 0:00:00
Mounted at /content/drive


Restaurando TAR do dataset:   0%|          | 0.00/861M [00:00<?, ?B/s]

Verificando TAR local:   0%|          | 0.00/861M [00:00<?, ?B/s]

Extraindo dataset local:   0%|          | 0/12004 [00:00<?, ?arq/s]

{
  "status": "DATASET_GPU_LOCAL_APROVADO",
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "amostras": 6000,
  "pacote_sha256": "163f46b536aa416c39c806490d6e6066b393bdb53a1e6de5911f24699a4b5434"
}


## 2. Contrato final antes do treino
Esta seção valida o contrato publicado no Notebook 08 e congela, antes dos novos
treinos, as regras de early stopping, checkpoint, seleção operacional e teste.
Regras fechadas:
- cada seed usa best.pt produzido pelo fitness de validação do Ultralytics 8.4.127;
- patience=20; teste e ASTA não participam do early stopping;
- o checkpoint operacional é escolhido somente na validação por F1 combinado em
  confidence=0,45, com desempates pré-definidos;
- o teste protegido avaliará os três checkpoints uma única vez e reportará a média,
  o desvio entre seeds e IC por bootstrap agrupado por fundo;
- resultados de teste nunca selecionarão seed, threshold ou checkpoint.

In [3]:
DRIVE_SELECTION=BACKUP_DIR/'experiments/yolo_family_selection_v3'/RUN_ID
DRIVE_FINAL=BACKUP_DIR/'experiments/yolo_final_multiseed_v3'/RUN_ID
DRIVE_FINAL.mkdir(parents=True,exist_ok=True)

def verificar_hash_canonico(registro,campo_hash):
    copia=dict(registro);observado=copia.pop(campo_hash)
    texto=json.dumps(copia,sort_keys=True,separators=(',',':'))
    calculado=hashlib.sha256(texto.encode()).hexdigest()
    assert calculado==observado,f'Hash canônico divergente em {campo_hash}'
    return observado

ponteiro=json.loads((BACKUP_DIR/'experiments/yolo_family_selection_v3/LATEST_SELECTION_V3.json').read_text(encoding='utf-8'))
contrato_selecao=json.loads((DRIVE_SELECTION/'contrato_selecao_yolo_v3.json').read_text(encoding='utf-8'))
config_selecao=json.loads((DRIVE_SELECTION/'protocolo_selecao_yolo_v3.json').read_text(encoding='utf-8'))
HASH_CONTRATO_SELECTION=verificar_hash_canonico(contrato_selecao,'sha256_contrato_selecao')
HASH_CONFIG_SELECTION=verificar_hash_canonico(config_selecao,'sha256_config_selecao')
assert ponteiro['run_id']==RUN_ID and ponteiro['sha256_manifest']==HASH_MANIFEST
assert ponteiro['sha256_contrato_selecao']==HASH_CONTRATO_SELECTION
assert contrato_selecao['aprovado'] is True
assert contrato_selecao['vencedor_recomendado']=='yolo11n_obb'
assert float(contrato_selecao['threshold_confianca'])==0.45
assert contrato_selecao['teste_sintetico_lido'] is False and contrato_selecao['asta_lido'] is False
assert config_selecao['ultralytics']=='8.4.127'

CONFIG_FINAL={
    'protocolo':'yolo_final_multiseed_v1','run_id':RUN_ID,
    'sha256_manifest':HASH_MANIFEST,
    'sha256_contrato_selecao':HASH_CONTRATO_SELECTION,
    'sha256_config_selecao':HASH_CONFIG_SELECTION,
    'familia':'yolo11n_obb','checkpoint_inicial':'yolo11n-obb.pt',
    'seeds':[20260823,20260824,20260825],
    'reutilizar_seed_selecao_se_identica':True,
    'treino':config_selecao['treino'],
    'early_stopping':{
        'conjunto':'val','patience':20,
        'checkpoint_por_seed':'best.pt do fitness interno de validação do Ultralytics 8.4.127',
        'teste_sintetico_usado':False,'asta_usado':False
    },
    'predicao_val':config_selecao['predicao_val'],
    'threshold_confianca_fixo':float(contrato_selecao['threshold_confianca']),
    'matching':config_selecao['matching'],
    'regra_checkpoint_operacional':[
        'maior_f1_combinado_val','menor_taxa_frames_negativos_com_fp_val',
        'maior_map50_95_obb_val','menor_seed'
    ],
    'plano_teste_protegido':{
        'evento_unico':True,'avaliar_tres_checkpoints':True,
        'resultado_primario':'média das métricas das três seeds',
        'variabilidade_treino':'desvio-padrão e faixa entre seeds',
        'incerteza_amostral':'bootstrap por sha256_fundo da média entre seeds',
        'selecionar_seed_com_teste':False,'recalibrar_threshold_com_teste':False,
        'checkpoint_operacional_para_asta':'escolhido somente na validação',
        'asta_permanece_intocado':True
    }
}
assert CONFIG_FINAL['treino']['patience']==CONFIG_FINAL['early_stopping']['patience']==20
assert CONFIG_FINAL['seeds']==contrato_selecao['seeds_finais']
texto_final=json.dumps(CONFIG_FINAL,sort_keys=True,separators=(',',':'))
HASH_CONFIG_FINAL=hashlib.sha256(texto_final.encode()).hexdigest()
CONFIG_FINAL['sha256_config_final']=HASH_CONFIG_FINAL
config_final_path=RUN/'protocolo_treino_final_multiseed_v3.json'
config_final_path.write_text(json.dumps(CONFIG_FINAL,indent=2),encoding='utf-8')
remoto=DRIVE_FINAL/config_final_path.name
if remoto.exists():
    assert json.loads(remoto.read_text(encoding='utf-8'))==CONFIG_FINAL,'Protocolo final remoto divergente.'
else:
    shutil.copy2(config_final_path,remoto)
print(json.dumps(CONFIG_FINAL,indent=2,ensure_ascii=False))
print('PROTOCOLO FINAL MULTISEED PUBLICADO. Teste sintético e ASTA seguem bloqueados.')

{
  "protocolo": "yolo_final_multiseed_v1",
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "sha256_contrato_selecao": "bc89537e86230ce1757ae990054859b59c59bbfcdfae8ef3daf0a4f13f5cd391",
  "sha256_config_selecao": "25af65bbb44f0b2d84c7064ecebe698dc81e74df842b85e1bcb2426d439ff0b1",
  "familia": "yolo11n_obb",
  "checkpoint_inicial": "yolo11n-obb.pt",
  "seeds": [
    20260823,
    20260824,
    20260825
  ],
  "reutilizar_seed_selecao_se_identica": true,
  "treino": {
    "epochs": 100,
    "patience": 20,
    "imgsz": 512,
    "batch": 16,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "cos_lr": true,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "deterministic": true,
    "amp": true,
    "workers": 2,
    "mosaic": 0.0,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.0,
    "degrees": 0.0,
    "translate": 0.0,
    "scale": 0.0,

## 3. Treinos independentes e retomáveis
Checkpoints parciais são publicados a cada cinco épocas. Uma seed concluída é
reutilizada somente se o registro, o hash da configuração e o hash do best.pt
coincidirem. Alterar hiperparâmetros exige um novo protocolo e outro diretório.

In [4]:
EXECUTAR_TREINOS_FINAIS=True
if not EXECUTAR_TREINOS_FINAIS:
    print('Treinos bloqueados. Compartilhe primeiro a saída da seção 2; depois altere a chave para True.')
else:
    import ultralytics
    from ultralytics import YOLO
    assert ultralytics.__version__=='8.4.127',f'Versão inesperada: {ultralytics.__version__}'
    LOCAL_PROJECT=Path('/content/yolo_final_multiseed_v3')/RUN_ID
    LOCAL_PROJECT.mkdir(parents=True,exist_ok=True)
    registros=[]

    selecao_treino=DRIVE_SELECTION/'treinos/yolo11n_obb'
    conclusao_selecao=json.loads((selecao_treino/'conclusao_treino.json').read_text(encoding='utf-8'))
    assert conclusao_selecao['seed']==20260823
    assert conclusao_selecao['sha256_config_selecao']==HASH_CONFIG_SELECTION
    assert conclusao_selecao['sha256_best_pt']==contrato_selecao['sha256_best_pt']
    assert sha256_arquivo(selecao_treino/'weights/best.pt')==conclusao_selecao['sha256_best_pt']
    HASH_PESO_INICIAL_ESPERADO=conclusao_selecao['sha256_peso_inicial']

    def callback_checkpoint(destino,seed):
        def salvar(trainer):
            epoca=int(trainer.epoch)+1
            if epoca%5!=0 and not trainer.stop:return
            destino.mkdir(parents=True,exist_ok=True)
            candidatos=[Path(trainer.last),Path(trainer.best),Path(trainer.save_dir)/'results.csv',Path(trainer.save_dir)/'args.yaml']
            for caminho in candidatos:
                if caminho.is_file():shutil.copy2(caminho,destino/caminho.name)
            estado={'seed':int(seed),'epoca':epoca,'sha256_config_final':HASH_CONFIG_FINAL,
                    'sha256_peso_inicial':HASH_PESO_INICIAL_ESPERADO}
            (destino/'estado_checkpoint.json').write_text(json.dumps(estado,indent=2),encoding='utf-8')
        return salvar

    for seed in CONFIG_FINAL['seeds']:
        chave=f'seed_{seed}'
        remoto_seed=DRIVE_FINAL/'treinos'/chave
        conclusao_path=remoto_seed/'conclusao_treino_final.json'
        if conclusao_path.exists():
            registro=json.loads(conclusao_path.read_text(encoding='utf-8'))
            assert registro['sha256_config_final']==HASH_CONFIG_FINAL and registro['seed']==seed
            best_path=BACKUP_DIR/registro['best_pt_relativo_drive']
            assert best_path.is_file() and sha256_arquivo(best_path)==registro['sha256_best_pt']
            registros.append(registro);print(chave,'já concluída e reutilizada.');continue

        remoto_seed.mkdir(parents=True,exist_ok=True)
        if seed==20260823:
            best=selecao_treino/'weights/best.pt';last=selecao_treino/'weights/last.pt'
            registro={
                'protocolo':'yolo_final_multiseed_v1','familia':'yolo11n_obb','seed':seed,
                'sha256_config_final':HASH_CONFIG_FINAL,'checkpoint_inicial':'yolo11n-obb.pt',
                'sha256_peso_inicial':HASH_PESO_INICIAL_ESPERADO,
                'sha256_best_pt':sha256_arquivo(best),'sha256_last_pt':sha256_arquivo(last),
                'best_pt_relativo_drive':str(best.relative_to(BACKUP_DIR)).replace('\\','/'),
                'epocas_registradas':int(conclusao_selecao['epocas_registradas']),
                'origem':'reutilizado_do_notebook_08_com_invariantes_verificados','concluido':True
            }
            conclusao_path.write_text(json.dumps(registro,indent=2),encoding='utf-8')
            registros.append(registro);print(json.dumps(registro,indent=2,ensure_ascii=False));continue

        checkpoint_remoto=remoto_seed/'checkpoint'
        ultimo_remoto=checkpoint_remoto/'last.pt'
        nome_execucao=f'yolo11n_obb_seed_{seed}'
        destino_local=LOCAL_PROJECT/nome_execucao
        if ultimo_remoto.exists():
            estado=json.loads((checkpoint_remoto/'estado_checkpoint.json').read_text(encoding='utf-8'))
            assert estado['seed']==seed and estado['sha256_config_final']==HASH_CONFIG_FINAL
            pesos_local=destino_local/'weights';pesos_local.mkdir(parents=True,exist_ok=True)
            for nome in ['last.pt','best.pt']:
                if (checkpoint_remoto/nome).exists():shutil.copy2(checkpoint_remoto/nome,pesos_local/nome)
            for nome in ['results.csv','args.yaml']:
                if (checkpoint_remoto/nome).exists():shutil.copy2(checkpoint_remoto/nome,destino_local/nome)
            modelo=YOLO(pesos_local/'last.pt')
            modelo.add_callback('on_model_save',callback_checkpoint(checkpoint_remoto,seed))
            print('Retomando',chave,'do checkpoint remoto.')
            modelo.train(resume=True,device=0,workers=CONFIG_FINAL['treino']['workers'])
        else:
            modelo=YOLO(CONFIG_FINAL['checkpoint_inicial'])
            peso_inicial=Path(CONFIG_FINAL['checkpoint_inicial'])
            assert peso_inicial.is_file()
            assert sha256_arquivo(peso_inicial)==HASH_PESO_INICIAL_ESPERADO
            modelo.add_callback('on_model_save',callback_checkpoint(checkpoint_remoto,seed))
            args={**CONFIG_FINAL['treino'],'data':str(dataset_yaml),'seed':seed,'device':0,
                  'project':str(LOCAL_PROJECT),'name':nome_execucao,'exist_ok':True,
                  'pretrained':True,'val':True,'plots':True,'save':True,'save_period':5,
                  'cache':False,'verbose':True}
            modelo.train(**args)

        pasta_resultado=Path(modelo.trainer.save_dir)
        best=pasta_resultado/'weights/best.pt';last=pasta_resultado/'weights/last.pt'
        assert best.is_file() and last.is_file() and (pasta_resultado/'results.csv').is_file()
        copiar_arvore_com_progresso(pasta_resultado,remoto_seed,f'Backup final {chave}')
        best_remoto=remoto_seed/'weights/best.pt';last_remoto=remoto_seed/'weights/last.pt'
        registro={
            'protocolo':'yolo_final_multiseed_v1','familia':'yolo11n_obb','seed':seed,
            'sha256_config_final':HASH_CONFIG_FINAL,'checkpoint_inicial':'yolo11n-obb.pt',
            'sha256_peso_inicial':HASH_PESO_INICIAL_ESPERADO,
            'sha256_best_pt':sha256_arquivo(best_remoto),'sha256_last_pt':sha256_arquivo(last_remoto),
            'best_pt_relativo_drive':str(best_remoto.relative_to(BACKUP_DIR)).replace('\\','/'),
            'epocas_registradas':int(len(pd.read_csv(remoto_seed/'results.csv'))),
            'origem':'treino_final_independente','concluido':True
        }
        conclusao_path.write_text(json.dumps(registro,indent=2),encoding='utf-8')
        registros.append(registro);print(json.dumps(registro,indent=2,ensure_ascii=False))
    print('TREINOS MULTISEED CONCLUÍDOS:',[r['seed'] for r in registros])

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
{
  "protocolo": "yolo_final_multiseed_v1",
  "familia": "yolo11n_obb",
  "seed": 20260823,
  "sha256_config_final": "457b70c8e96176827bf4a3f755963e6c6c3e607bec7d8133233518b26be27870",
  "checkpoint_inicial": "yolo11n-obb.pt",
  "sha256_peso_inicial": "b62898ebf38940ca4df323863e45ee9d84a1a46d5d11ebdde529fb33aa9f3a32",
  "sha256_best_pt": "dab6ececdc1011a645bfa4a3486ba3bd10746d083fa3ccc61ac462fc21b1aa7e",
  "sha256_last_pt": "a0d6bb6c3a7bf5b48991834e8f0cfe2e6f3bd1bf286d1a211f5172ac55f5ff21",
  "best_pt_relativo_drive": "experiments/yolo_family_selection_v3/synthetic_v3_c41d5091de56/treinos/yolo11n_obb/weights/best.pt",
  "epocas_registradas": 41,
  "origem": "reutilizado_do_notebook

Backup final seed_20260824:   0%|          | 0/31 [00:00<?, ?arq/s]

{
  "protocolo": "yolo_final_multiseed_v1",
  "familia": "yolo11n_obb",
  "seed": 20260824,
  "sha256_config_final": "457b70c8e96176827bf4a3f755963e6c6c3e607bec7d8133233518b26be27870",
  "checkpoint_inicial": "yolo11n-obb.pt",
  "sha256_peso_inicial": "b62898ebf38940ca4df323863e45ee9d84a1a46d5d11ebdde529fb33aa9f3a32",
  "sha256_best_pt": "a2b64773f32868ac952d81c33b331f1759b2250a3a04f49d5ab105f7bb79dd0c",
  "sha256_last_pt": "4673b4b87645993582b4fbb189855b0b960f8fccd8baf8846e432ed70383c836",
  "best_pt_relativo_drive": "experiments/yolo_final_multiseed_v3/synthetic_v3_c41d5091de56/treinos/seed_20260824/weights/best.pt",
  "epocas_registradas": 49,
  "origem": "treino_final_independente",
  "concluido": true
}
New https://pypi.org/project/ultralytics/8.4.132 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.127 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment

Backup final seed_20260825:   0%|          | 0/31 [00:00<?, ?arq/s]

{
  "protocolo": "yolo_final_multiseed_v1",
  "familia": "yolo11n_obb",
  "seed": 20260825,
  "sha256_config_final": "457b70c8e96176827bf4a3f755963e6c6c3e607bec7d8133233518b26be27870",
  "checkpoint_inicial": "yolo11n-obb.pt",
  "sha256_peso_inicial": "b62898ebf38940ca4df323863e45ee9d84a1a46d5d11ebdde529fb33aa9f3a32",
  "sha256_best_pt": "68433cd33a9fb77523ac0adf5fed918edae7f1bba4b17abf8618b119f89f349d",
  "sha256_last_pt": "17cb576ec79cc0779dee25050ea3ad13bff4bc30c3e89bed43085bdffb37b399",
  "best_pt_relativo_drive": "experiments/yolo_final_multiseed_v3/synthetic_v3_c41d5091de56/treinos/seed_20260825/weights/best.pt",
  "epocas_registradas": 48,
  "origem": "treino_final_independente",
  "concluido": true
}
TREINOS MULTISEED CONCLUÍDOS: [20260823, 20260824, 20260825]


## 4. Gate local dos três checkpoints
Execute esta seção antes da avaliação de validação. Compartilhe o JSON produzido.

In [5]:
registros=[];erros=[]
for seed in CONFIG_FINAL['seeds']:
    caminho=DRIVE_FINAL/'treinos'/f'seed_{seed}'/'conclusao_treino_final.json'
    if not caminho.exists():
        erros.append(f'ausente:{seed}');continue
    registro=json.loads(caminho.read_text(encoding='utf-8'))
    if registro.get('sha256_config_final')!=HASH_CONFIG_FINAL:erros.append(f'config:{seed}')
    best=BACKUP_DIR/registro['best_pt_relativo_drive']
    if not best.is_file() or sha256_arquivo(best)!=registro.get('sha256_best_pt'):erros.append(f'best:{seed}')
    registros.append(registro)
resumo_treinos={
    'status':'TREINOS_MULTISEED_APROVADOS' if not erros and len(registros)==3 else 'REPROVADO',
    'run_id':RUN_ID,'sha256_manifest':HASH_MANIFEST,'sha256_config_final':HASH_CONFIG_FINAL,
    'seeds_encontradas':sorted(r['seed'] for r in registros),
    'epocas_por_seed':{str(r['seed']):r['epocas_registradas'] for r in registros},
    'sha256_best_por_seed':{str(r['seed']):r['sha256_best_pt'] for r in registros},
    'erros':erros
}
print(json.dumps(resumo_treinos,indent=2,ensure_ascii=False))
assert resumo_treinos['status']=='TREINOS_MULTISEED_APROVADOS'

{
  "status": "TREINOS_MULTISEED_APROVADOS",
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "sha256_config_final": "457b70c8e96176827bf4a3f755963e6c6c3e607bec7d8133233518b26be27870",
  "seeds_encontradas": [
    20260823,
    20260824,
    20260825
  ],
  "epocas_por_seed": {
    "20260823": 41,
    "20260824": 49,
    "20260825": 48
  },
  "sha256_best_por_seed": {
    "20260823": "dab6ececdc1011a645bfa4a3486ba3bd10746d083fa3ccc61ac462fc21b1aa7e",
    "20260824": "a2b64773f32868ac952d81c33b331f1759b2250a3a04f49d5ab105f7bb79dd0c",
    "20260825": "68433cd33a9fb77523ac0adf5fed918edae7f1bba4b17abf8618b119f89f349d"
  },
  "erros": []
}


## 5. Avaliação comum somente na validação
Esta célula usa o threshold 0,45 já congelado; não recalibra uma grade por seed.
Predições são processadas em streaming com batch=1 para limitar VRAM. A recomendação
de checkpoint operacional ainda precisa de aprovação humana na seção seguinte.

In [6]:
EXECUTAR_AVALIACAO_FINAL_VAL=True
if not EXECUTAR_AVALIACAO_FINAL_VAL:
    print('Avaliação bloqueada. Execute somente após o gate TREINOS_MULTISEED_APROVADOS.')
else:
    from ultralytics import YOLO
    import gc,torch
    gc.collect();torch.cuda.empty_cache()

    def parametros_linha(p0,p1):
        p0,p1=np.asarray(p0,dtype=float),np.asarray(p1,dtype=float)
        v=p1-p0;comp=float(np.linalg.norm(v));direcao=v/comp if comp>0 else np.array([1.,0.])
        return (p0+p1)/2,direcao,comp
    def centerline(p):
        p=np.asarray(p,dtype=float);d01=np.linalg.norm(p[0]-p[1]);d12=np.linalg.norm(p[1]-p[2])
        return ((p[0]+p[3])/2,(p[1]+p[2])/2) if d01>=d12 else ((p[0]+p[1])/2,(p[2]+p[3])/2)
    def iou_poligonos(a,b):
        a,b=np.asarray(a,np.float32),np.asarray(b,np.float32);aa=abs(float(cv2.contourArea(a)));ab=abs(float(cv2.contourArea(b)))
        inter=float(cv2.intersectConvexConvex(a,b)[0]);uniao=aa+ab-inter
        return inter/uniao if uniao>0 else 0.0
    def metricas_match(pred,gt):
        p0,p1=centerline(pred);g0,g1=centerline(gt);cp,dp,_=parametros_linha(p0,p1);cg,dg,lg=parametros_linha(g0,g1)
        ang=float(np.degrees(np.arccos(np.clip(abs(np.dot(dp,dg)),-1,1))))
        v=cp-cg;dist=float(np.linalg.norm(v-np.dot(v,dg)*dg))
        t=sorted([np.dot(p0-cg,dg),np.dot(p1-cg,dg)]);inter=max(0,min(t[1],lg/2)-max(t[0],-lg/2));cob=float(inter/lg) if lg>0 else 0.0
        return {'iou':iou_poligonos(pred,gt),'angulo':ang,'distancia':dist,'cobertura':cob}
    def ler_gt(linha):
        tokens=(RUN/'labels'/linha.split/(linha.id+'.txt')).read_text(encoding='utf-8').split()
        return None if not tokens else np.asarray([float(x) for x in tokens[1:9]]).reshape(4,2)*np.array([512.,512.])
    def aceito(m,lim):
        return m['iou']>=lim['iou'] or (m['angulo']<=lim['angulo'] and m['distancia']<=lim['distancia'] and m['cobertura']>=lim['cobertura'])
    def avaliar(dados,predicoes,threshold):
        saida=[];lim=CONFIG_FINAL['matching']
        for linha in dados.itertuples():
            grupo=predicoes[(predicoes.id==linha.id)&(predicoes.conf>=threshold)]
            polys=[g[['x1','y1','x2','y2','x3','y3','x4','y4']].to_numpy(float).reshape(4,2) for _,g in grupo.iterrows()]
            gt=ler_gt(linha);match=False
            if gt is not None:match=any(aceito(metricas_match(p,gt),lim) for p in polys)
            saida.append({'id':linha.id,'tipo':linha.tipo,'subtipo':linha.subtipo,'sha256_fundo':linha.sha256_fundo,
                          'tp':int(gt is not None and match),'fp':len(polys)-int(gt is not None and match),
                          'fn':int(gt is not None and not match),'n_predicoes':len(polys)})
        out=pd.DataFrame(saida);tp=int(out.tp.sum());fp=int(out.fp.sum());fn=int(out.fn.sum())
        p=tp/(tp+fp) if tp+fp else 0;r=tp/(tp+fn) if tp+fn else 0;f1=2*p*r/(p+r) if p+r else 0
        neg=out[out.tipo=='negativo']
        return out,{'TP':tp,'FP':fp,'FN':fn,'precisao':p,'recall':r,'f1':f1,
                    'fp_por_imagem_negativa':float(neg.fp.mean()),
                    'taxa_frames_negativos_com_fp':float((neg.fp>0).mean())}

    df_val=df[df.split=='val'].copy().sort_values('id');ids_esperados=set(df_val.id.astype(str))
    pasta_val=RUN/'images/val';assert {p.stem for p in pasta_val.glob('*.png')}==ids_esperados
    ARTEFATOS_VAL=RUN/'yolo_final_validation';ARTEFATOS_VAL.mkdir(exist_ok=True)
    linhas=[]
    for registro in registros:
        seed=int(registro['seed']);best=BACKUP_DIR/registro['best_pt_relativo_drive']
        assert sha256_arquivo(best)==registro['sha256_best_pt']
        gc.collect();torch.cuda.empty_cache();modelo=YOLO(best)
        preds=[];tempos=[];ids=[]
        resultados=modelo.predict(source=str(pasta_val),stream=True,batch=1,imgsz=CONFIG_FINAL['treino']['imgsz'],
            conf=CONFIG_FINAL['predicao_val']['conf_min'],iou=CONFIG_FINAL['predicao_val']['iou_nms'],
            max_det=CONFIG_FINAL['predicao_val']['max_det'],device=0,verbose=False)
        for resultado in tqdm(resultados,total=len(ids_esperados),desc=f'Val seed {seed}',unit='img'):
            id_amostra=Path(resultado.path).stem;ids.append(id_amostra);tempos.append(float(resultado.speed.get('inference',np.nan)))
            if resultado.obb is None:continue
            for poly,conf in zip(resultado.obb.xyxyxyxy.cpu().numpy(),resultado.obb.conf.cpu().numpy()):
                plano=poly.reshape(-1);preds.append({'id':id_amostra,'conf':float(conf),
                    **{f'{e}{i+1}':float(plano[2*i+(0 if e=='x' else 1)]) for i in range(4) for e in ['x','y']}})
        assert len(ids)==len(ids_esperados) and len(set(ids))==len(ids) and set(ids)==ids_esperados
        pred=pd.DataFrame(preds,columns=['id','conf','x1','y1','x2','y2','x3','y3','x4','y4'])
        pred.to_csv(ARTEFATOS_VAL/f'predicoes_val_seed_{seed}.csv',index=False)
        por_imagem,resumo=avaliar(df_val,pred,CONFIG_FINAL['threshold_confianca_fixo'])
        por_imagem.to_csv(ARTEFATOS_VAL/f'resultado_val_seed_{seed}.csv',index=False)
        metricas=modelo.val(data=str(dataset_yaml),split='val',imgsz=CONFIG_FINAL['treino']['imgsz'],
            batch=CONFIG_FINAL['treino']['batch'],device=0,workers=2,plots=False,verbose=False,
            project=str(ARTEFATOS_VAL/'ultralytics'),name=f'seed_{seed}',exist_ok=True)
        linhas.append({'seed':seed,**resumo,'map50_95_obb':float(metricas.box.map),
                       'map50_obb':float(metricas.box.map50),'inferencia_ms_media':float(np.nanmean(tempos)),
                       'sha256_best_pt':registro['sha256_best_pt']})
        del resultados,resultado,metricas,modelo;gc.collect();torch.cuda.empty_cache()

    comparacao=pd.DataFrame(linhas).sort_values('seed').reset_index(drop=True)
    assert set(comparacao.seed)==set(CONFIG_FINAL['seeds']) and len(comparacao)==3
    ordenada=comparacao.sort_values(['f1','taxa_frames_negativos_com_fp','map50_95_obb','seed'],ascending=[False,True,False,True])
    campeao=ordenada.iloc[0]
    comparacao_path=ARTEFATOS_VAL/'comparacao_seeds_val_v3.csv';comparacao.to_csv(comparacao_path,index=False)
    recomendacao={
        'protocolo':'yolo_final_multiseed_v1','run_id':RUN_ID,'sha256_manifest':HASH_MANIFEST,
        'sha256_config_final':HASH_CONFIG_FINAL,'threshold_confianca':CONFIG_FINAL['threshold_confianca_fixo'],
        'seeds_avaliadas':CONFIG_FINAL['seeds'],'checkpoint_operacional_seed':int(campeao.seed),
        'checkpoint_operacional_sha256':campeao.sha256_best_pt,
        'regra_checkpoint_operacional':CONFIG_FINAL['regra_checkpoint_operacional'],
        'sha256_tabela_validacao':sha256_arquivo(comparacao_path),
        'teste_sintetico_lido':False,'asta_lido':False
    }
    (ARTEFATOS_VAL/'recomendacao_checkpoint_operacional_v3.json').write_text(json.dumps(recomendacao,indent=2),encoding='utf-8')
    copiar_arvore_com_progresso(ARTEFATOS_VAL,DRIVE_FINAL/'avaliacao_val','Backup avaliação multiseed em validação')
    print(comparacao[['seed','precisao','recall','f1','taxa_frames_negativos_com_fp','map50_95_obb','inferencia_ms_media']])
    print(json.dumps(recomendacao,indent=2,ensure_ascii=False))
    print('RECOMENDAÇÃO LOCAL PRONTA. Teste e ASTA continuam bloqueados.')

Val seed 20260823:   0%|          | 0/900 [00:00<?, ?img/s]

Ultralytics 8.4.127 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2237.4±555.6 MB/s, size: 135.6 KB)
val: Scanning /content/data/synthetic_runs/synthetic_v3_c41d5091de56/labels/val.cache... 900 images, 450 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 900/900 235.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 6.1it/s 9.4s
                   all        900        450      0.905      0.958      0.977      0.551
Speed: 1.1ms preprocess, 3.3ms inference, 0.0ms loss, 2.8ms postprocess per image


Val seed 20260824:   0%|          | 0/900 [00:00<?, ?img/s]

Ultralytics 8.4.127 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2648.5±666.4 MB/s, size: 122.0 KB)
val: Scanning /content/data/synthetic_runs/synthetic_v3_c41d5091de56/labels/val.cache... 900 images, 450 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 900/900 314.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 5.7it/s 10.0s
                   all        900        450      0.975      0.939      0.966      0.509
Speed: 1.1ms preprocess, 3.1ms inference, 0.0ms loss, 3.8ms postprocess per image


Val seed 20260825:   0%|          | 0/900 [00:00<?, ?img/s]

Ultralytics 8.4.127 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2160.4±640.9 MB/s, size: 125.1 KB)
val: Scanning /content/data/synthetic_runs/synthetic_v3_c41d5091de56/labels/val.cache... 900 images, 450 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 900/900 209.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 5.2it/s 10.9s
                   all        900        450      0.954      0.936      0.973      0.637
Speed: 1.4ms preprocess, 3.2ms inference, 0.0ms loss, 2.8ms postprocess per image


Backup avaliação multiseed em validação:   0%|          | 0/8 [00:00<?, ?arq/s]

       seed  precisao    recall        f1  taxa_frames_negativos_com_fp  \
0  20260823  0.940678  0.986667  0.963124                      0.002222   
1  20260824  0.982379  0.991111  0.986726                      0.000000   
2  20260825  0.941935  0.973333  0.957377                      0.004444   

   map50_95_obb  inferencia_ms_media  
0      0.550573            11.114018  
1      0.508984            10.948081  
2      0.636749            11.419466  
{
  "protocolo": "yolo_final_multiseed_v1",
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "sha256_config_final": "457b70c8e96176827bf4a3f755963e6c6c3e607bec7d8133233518b26be27870",
  "threshold_confianca": 0.45,
  "seeds_avaliadas": [
    20260823,
    20260824,
    20260825
  ],
  "checkpoint_operacional_seed": 20260824,
  "checkpoint_operacional_sha256": "a2b64773f32868ac952d81c33b331f1759b2250a3a04f49d5ab105f7bb79dd0c",
  "regra_checkpoint_operaciona

## 6. Gate humano e contrato dos modelos finais
Só publique após revisar a tabela das três seeds. Este gate congela os três hashes,
o checkpoint operacional e todo o plano do teste. Ele ainda não executa o teste.

In [7]:
APROVAR_MODELOS_FINAIS=True
recomendacao_path=RUN/'yolo_final_validation/recomendacao_checkpoint_operacional_v3.json'
if not APROVAR_MODELOS_FINAIS:
    print('Contrato final não publicado. Compartilhe a tabela e a recomendação para revisão.')
else:
    recomendacao=json.loads(recomendacao_path.read_text(encoding='utf-8'))
    assert recomendacao['sha256_manifest']==HASH_MANIFEST and recomendacao['sha256_config_final']==HASH_CONFIG_FINAL
    tabela_path=RUN/'yolo_final_validation/comparacao_seeds_val_v3.csv'
    assert sha256_arquivo(tabela_path)==recomendacao['sha256_tabela_validacao']
    checkpoints=[]
    for registro in registros:
        best=BACKUP_DIR/registro['best_pt_relativo_drive']
        assert sha256_arquivo(best)==registro['sha256_best_pt']
        checkpoints.append({'seed':registro['seed'],'sha256_best_pt':registro['sha256_best_pt'],
            'best_pt_relativo_drive':registro['best_pt_relativo_drive']})
    assert len(checkpoints)==3 and len({x['seed'] for x in checkpoints})==3
    assert recomendacao['checkpoint_operacional_sha256'] in {x['sha256_best_pt'] for x in checkpoints}
    contrato={**recomendacao,'aprovado':True,'checkpoints':sorted(checkpoints,key=lambda x:x['seed']),
              'plano_teste_protegido':CONFIG_FINAL['plano_teste_protegido']}
    texto=json.dumps(contrato,sort_keys=True,separators=(',',':'))
    contrato['sha256_contrato_modelos_finais']=hashlib.sha256(texto.encode()).hexdigest()
    destino=DRIVE_FINAL/'contrato_modelos_finais_v3.json'
    if destino.exists():assert json.loads(destino.read_text(encoding='utf-8'))==contrato,'Contrato final remoto divergente.'
    else:destino.write_text(json.dumps(contrato,indent=2),encoding='utf-8')
    ponteiro=BACKUP_DIR/'experiments/yolo_final_multiseed_v3/LATEST_FINAL_MODELS_V3.json'
    ponteiro.parent.mkdir(parents=True,exist_ok=True)
    ponteiro.write_text(json.dumps({'run_id':RUN_ID,'sha256_manifest':HASH_MANIFEST,
        'sha256_contrato_modelos_finais':contrato['sha256_contrato_modelos_finais'],
        'checkpoint_operacional_seed':contrato['checkpoint_operacional_seed']},indent=2),encoding='utf-8')
    print(json.dumps(contrato,indent=2,ensure_ascii=False))
    print('MODELOS FINAIS PUBLICADOS. Notebook de teste pode ser gerado; ASTA permanece bloqueado.')

{
  "protocolo": "yolo_final_multiseed_v1",
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "sha256_config_final": "457b70c8e96176827bf4a3f755963e6c6c3e607bec7d8133233518b26be27870",
  "threshold_confianca": 0.45,
  "seeds_avaliadas": [
    20260823,
    20260824,
    20260825
  ],
  "checkpoint_operacional_seed": 20260824,
  "checkpoint_operacional_sha256": "a2b64773f32868ac952d81c33b331f1759b2250a3a04f49d5ab105f7bb79dd0c",
  "regra_checkpoint_operacional": [
    "maior_f1_combinado_val",
    "menor_taxa_frames_negativos_com_fp_val",
    "maior_map50_95_obb_val",
    "menor_seed"
  ],
  "sha256_tabela_validacao": "1ebf261bd0e66cf610a85b96927357ea966a2e44211a85f546cde98666955f42",
  "teste_sintetico_lido": false,
  "asta_lido": false,
  "aprovado": true,
  "checkpoints": [
    {
      "seed": 20260823,
      "sha256_best_pt": "dab6ececdc1011a645bfa4a3486ba3bd10746d083fa3ccc61ac462fc21b1aa7e",
      "bes